#Schritt 5: Risiko-Klassifikation

Songs werden zu vier Kategorien zugeordnet: kein_hinweis, ästhetische_melancholie, herzschmerz, möglicher_risikohinweis.

In [ ]:
import pandas as pd
EMOTIONEN_TRAURIG = ['sadness', 'grief', 'disappointment', 'remorse']
SCHLÜSSELWÖRTER_RISIKOHINWEIS = ['hopeless', 'no way out', 'give up', 'give in', 'no strength left', 'disappear', 'nobody hears me', 'pointless', 'the end', 'cant go on', "can't go on", 'worthless']
SCHLÜSSELWÖRTER_HERZSCHMERZ = ['heartbreak', 'miss you', "you're gone", 'youre gone', 'broken heart', 'left me', 'lonely without you', 'without you']
SCHLÜSSELWÖRTER_NOSTALGIE = ['i remember', 'back then', 'old photos', 'those days', 'nostalgia', 'used to']

In [ ]:
def zeige_validierungs_warnung():
    print('Vorsicht: Diese Risiko-Klassifikation ist nicht psychologisch validiert. Sie dient nur als technischer Anhaltspunkt für die Forschung und darf nicht als Diagnose oder als Aussage über eine reale Person verwendet werden.')

In [ ]:
def enthält_schlüsselwort(text, wörter_liste):
    for wort in wörter_liste:
        if wort in text:
            return True
    return False

In [ ]:
def klassifiziere_risiko(text, top_emotion=None, vader_kategorie=None):
    if enthält_schlüsselwort(text, SCHLÜSSELWÖRTER_RISIKOHINWEIS):
        return 'möglicher_risikohinweis'
    if enthält_schlüsselwort(text, SCHLÜSSELWÖRTER_HERZSCHMERZ):
        return 'herzschmerz'
    if enthält_schlüsselwort(text, SCHLÜSSELWÖRTER_NOSTALGIE):
        return 'ästhetische_melancholie'
    if top_emotion in EMOTIONEN_TRAURIG and vader_kategorie == 'negativ':
        return 'ästhetische_melancholie'
    return 'kein_hinweis'

In [ ]:
def klassifiziere_datensatz(daten, text_spalte='lyrics_clean'):
    risiko_liste = []
    for index, zeile in daten.iterrows():
        text = zeile[text_spalte]
        top_emotion = zeile['top_emotion'] if 'top_emotion' in daten.columns else None
        vader_kategorie = zeile['vader_kategorie'] if 'vader_kategorie' in daten.columns else None
        risiko_label = klassifiziere_risiko(text, top_emotion, vader_kategorie)
        risiko_liste.append(risiko_label)
    daten['risiko_label'] = risiko_liste
    return daten

In [ ]:
def klassifiziere_segmente(segment_daten):
    risiko_liste = []
    for index, zeile in segment_daten.iterrows():
        text = zeile['segment_text_clean']
        risiko_label = klassifiziere_risiko(text)
        risiko_liste.append(risiko_label)
    segment_daten['segment_risiko_label'] = risiko_liste
    return segment_daten

#Rangfolge
Für die Aggregation von Abschnitts- zu Song-Ebene. Ein Song bekommt die ernsteste Kategorie, die in irgendeinem seiner Abschnitte vorkommt. Ist jedoch eine diskutable Entscheidung, denn theoretisch färbt ein einzelner auffälliger Chorus  damit den ganzen Song ein, auch wenn alle anderen Abschnitte unauffällig sind. 

In [ ]:
RANGFOLGE = ['kein_hinweis', 'ästhetische_melancholie', 'herzschmerz', 'möglicher_risikohinweis']

In [ ]:
def aggregiere_segmente_zu_song(segment_daten):
    ergebnis_liste = []
    for song_id in segment_daten['song_id'].unique():
        segmente_des_songs = segment_daten[segment_daten['song_id'] == song_id]
        schwerste_kategorie = 'kein_hinweis'
        for label in segmente_des_songs['segment_risiko_label']:
            if RANGFOLGE.index(label) > RANGFOLGE.index(schwerste_kategorie):
                schwerste_kategorie = label
        ergebnis_liste.append({'song_id': song_id, 'title': segmente_des_songs['title'].iloc[0], 'risiko_label_aus_segmenten': schwerste_kategorie})
    return pd.DataFrame(ergebnis_liste)

## Pipeline ausführen

In [ ]:
zeige_validierungs_warnung()

#Song-Ebene

In [ ]:
try:
    daten = pd.read_csv('data/daten_mit_emotionen.csv')
except FileNotFoundError:
    daten = pd.read_csv('data/daten_mit_vader.csv')
daten = klassifiziere_datensatz(daten)
print('\nRisiko-Labels auf Song-Ebene')
print(daten[['title', 'risiko_label']])
daten.to_csv('data/daten_mit_risiko.csv', index=False)
print('Gespeichert unter: data/daten_mit_risiko.csv')

#Abschnitts-Ebene 

In [ ]:
segment_daten = pd.read_csv('data/segmente_vorverarbeitet.csv')
segment_daten = klassifiziere_segmente(segment_daten)
print('\nRisiko-Labels auf Abschnitts-Ebene (Verse/Chorus)')
print(segment_daten[['title', 'segment_typ', 'segment_risiko_label']])
song_labels_aus_segmenten = aggregiere_segmente_zu_song(segment_daten)
print('\nZu Song-Ebene aggregierte Abschnitts-Labels')
print(song_labels_aus_segmenten)
segment_daten.to_csv('data/segmente_mit_risiko.csv', index=False)
song_labels_aus_segmenten.to_csv('data/song_labels_aus_segmenten.csv', index=False)
print('\nGespeichert unter: data/segmente_mit_risiko.csv')
print('Gespeichert unter: data/song_labels_aus_segmenten.csv')